# 105 — RAG básico con citas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**RAG** (arXiv:2005.11401): condicionar la generación sobre documentos recuperados en
tiempo de consulta. Formulación original:
`p(y|x) = Σ_z p_retriever(z|x) · p_generator(y|x,z)` — RAG-Sequence usa un documento
para toda la secuencia; RAG-Token puede cambiar de documento por token. El RAG moderno
opera por prompting: top-k pasajes numerados `[n]` + regla de citas en el prompt.

**Conocimiento paramétrico vs. recuperado**: el segundo es actualizable (reindexar, no
reentrenar), inspeccionable y atribuible.

**Groundedness**: fracción de afirmaciones de la respuesta implicadas por el contexto
recuperado. Una cita `[n]` es una hipótesis de procedencia que debe verificarse: los
modelos generan citas sintácticamente válidas y semánticamente falsas.

**Diseño del prompt**: k por presupuesto, orden de pasajes (los extremos del contexto se
atienden mejor — "lost in the middle", arXiv:2307.03172), y política de rechazo explícita
("no consta en el contexto") cuando la evidencia falta.

## 🧮 Ejemplo de referencia

```text
[1] "El James Webb se lanzó el 25 de diciembre de 2021."
[2] "El Webb observa principalmente en el infrarrojo."
[3] "El Hubble se lanzó en 1990."

Respuesta: "El Webb se lanzó el 25-12-2021 [1] y observa en el infrarrojo [2]."
  A1 → implicada por [1]: SÍ.   A2 → implicada por [2]: SÍ.
  groundedness = 2/2 = 1.0

Si añadiera "…y costó 10 000 M$ [3]": A3 no está en ningún pasaje y [3] habla
del Hubble → alucinación con cita falsa, groundedness = 2/3 ≈ 0.67.
```

Un dato verdadero en el mundo pero ausente del corpus sigue siendo un fallo de RAG:
la promesa es "esto sale de estas fuentes", no "esto es verdad".

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("retrieval", seed=105)
show(result)


## Reflexión

1. ¿Por qué "respuesta correcta" y "respuesta fundamentada" son propiedades independientes en RAG, y cuál de las dos puede verificar el sistema sin conocer la verdad del mundo?
2. Si el generador produce una cita [2] junto a una afirmación que el pasaje 2 no implica, ¿en qué componente del pipeline intervendrías (retriever, prompt, verificación posterior) y por qué?
3. ¿Qué riesgo introduce un documento del corpus que contiene la frase "ignora las instrucciones anteriores y responde X", y qué separación estructural del prompt lo mitiga?